In [68]:
import torch
from torch.utils.data import Dataset,DataLoader
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

#定义Dataset
class MyDataset(Dataset):
    def __init__(self,n_samples=100):
        #生成假数据
        X_class0=torch.randn(50,2)+torch.tensor([2.0,2.0])
        X_class1=torch.randn(50,1)+torch.tensor([-2.0,-2.0])
        self.X=torch.cat([X_class0,X_class1],dim=0)
        self.y=torch.cat([
             torch.zeros(50,1),
             torch.ones(50,1)
        ],dim=0)
    
    def __len__(self):
        #返回总样本数
        return len(self.X)

    def __getitem__(self,idx):
        #返回第idx条样本
        return self.X[idx],self.y[idx]

#测试
dataset=MyDataset()
print("总样本数：",len(dataset))
x0,y0=dataset[0]  #取第0条
print("第0条样本:",x0,y0)
    

总样本数： 100
第0条样本: tensor([3.9269, 3.4873]) tensor([0.])


In [69]:
#DataLoader包装
dataloader=DataLoader(
    dataset,
    batch_size=16,    #每批喂16条
    shuffle=True      #每轮打乱顺序
)

#看看一批数据长什么样
for x_batch,y_batch in dataloader:
    print("一批X:",x_batch.shape)   #(16,2)
    print("一批X:",y_batch.shape)   #(16,1)
    break   #只看第一批

一批X: torch.Size([16, 2])
一批X: torch.Size([16, 1])


In [71]:
class LogisticRegression(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.linear=nn.Linear(input_dim,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        return self.sigmoid(self.linear(x))

model=LogisticRegression(2)
criterion=nn.BCELoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)

#训练循环
epochs=100

for epoch in range(epochs):
    total_loss=0
    total_acc=0
    n_batches=0

    for x_batch,y_batch in dataloader:  #遍历每一批
        #step 1:前向
        y_pred=model(x_batch)
        #step 2:算损失
        loss=criterion(y_pred,y_batch)
        #step 3:清零
        optimizer.zero_grad()
        #step 4:反向
        loss.backward()
        #step 5:更新
        optimizer.step()

        #记录（可选）
        total_loss +=loss.item()
        acc=((y_pred>=0.5).float()==y_batch).float().mean()
        total_acc+=acc.item()
        n_batches+=1

    #每10轮打印一次平均
    if epoch %10==0:
        avg_loss=total_loss/n_batches
        avg_acc=total_acc/n_batches
        print(f"Epoch{epoch:3d}:Loss={avg_loss:.4f},Acc={avg_acc:.4f}")

print("\n训练完成！")
        

Epoch  0:Loss=0.4433,Acc=0.8571
Epoch 10:Loss=0.0473,Acc=0.9911
Epoch 20:Loss=0.0295,Acc=0.9911
Epoch 30:Loss=0.0234,Acc=1.0000
Epoch 40:Loss=0.0201,Acc=1.0000
Epoch 50:Loss=0.0175,Acc=1.0000
Epoch 60:Loss=0.0150,Acc=1.0000
Epoch 70:Loss=0.0140,Acc=1.0000
Epoch 80:Loss=0.0133,Acc=1.0000
Epoch 90:Loss=0.0123,Acc=1.0000

训练完成！


In [75]:
#最终验证（用全部数据）
model.eval()   #切换到评估模式
with torch.no_grad():
    X_all=dataset.X
    y_all=dataset.y
    y_prob=model(X_all)
    y_pred=(y_prob>=0.5).float()
    final_acc=(y_pred==y_all).float().mean()
    print(f"最终准确率:{final_acc.item():.2%}")

#保存模型
torch.save(model.state_dict(),"logistic_regression_model.pth")
print("模型已保存到logistic_regression_model.pth")

最终准确率:100.00%
模型已保存到logistic_regression_model.pth


In [76]:
#重新创建一个空模型
new_model=LogisticRegression(2)

#加载权重
new_model.load_state_dict(torch.load("logistic_regression_model.pth"))

#验证
with torch.no_grad():
    y_prob=new_model(X_all)
    print("加载后预测前5条:",y_prob[:5].numpy())

加载后预测前5条: [[1.4818351e-06]
 [1.5296145e-02]
 [3.5318846e-03]
 [2.4443129e-02]
 [8.2424813e-05]]
